# OthelloRL Sweep Agent
Connects to the existing wandb sweep and runs trials on Colab hardware.

In [ ]:
# 1. Clone the repo
!git clone https://github.com/sanasuri101/OthelloRL.git
%cd OthelloRL

In [ ]:
# 2. Install dependencies
# Force-reinstall numpy to ensure Python files and C extensions match
!pip install -q --force-reinstall "numpy<2.0"
# Force-reinstall wandb to get wandb-core binary
!pip install -q --force-reinstall wandb
# Recompile pufferlib from source against the installed numpy (no cached wheels)
!pip install -q --no-binary pufferlib pufferlib

# Build the C extension without raylib
%cd othello
!make NO_RENDER=1
%cd ..

print("✓ All installs done. NOW RESTART THE RUNTIME: Runtime → Restart session")
print("After restart, run cells 3 onwards (skip this cell)")


In [ ]:
# 3. Verify everything imports correctly
import sys, os
sys.path.insert(0, '/content/OthelloRL')
sys.path = [p for p in sys.path if not p.endswith('/othello')]

import torch
import pufferlib
import wandb
print(f'torch: {torch.__version__}')
print(f'pufferlib: {pufferlib.__version__}')
print(f'wandb: {wandb.__version__}')
from othello import binding
print(f'CUDA available: {torch.cuda.is_available()}')
print('binding: OK')

In [ ]:
# 4. Log in to wandb using Colab secret
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["WANDB_AGENT_DISABLE_FLAPPING"] = "true"
os.chdir('/content/OthelloRL')

wandb.login()

In [ ]:
# 5. Create a new sweep (run only once — skip if you already have a sweep ID)
!wandb sweep /content/OthelloRL/sweep.yaml

In [ ]:
# 6. Run 3 parallel agents using multiprocessing (each gets its own wandb-core instance)
import multiprocessing as mp
import os

SWEEP_ID = "sriramanasuri-georgia-institute-of-technology/Connect4RL-othello/ux3575ja"
API_KEY = os.environ["WANDB_API_KEY"]

def worker(api_key, sweep_id):
    import os, sys, wandb
    os.chdir('/content/OthelloRL')
    sys.path.insert(0, '/content/OthelloRL')
    sys.path = [p for p in sys.path if not p.endswith('/othello')]
    os.environ["WANDB_API_KEY"] = api_key
    os.environ["WANDB_AGENT_DISABLE_FLAPPING"] = "true"
    wandb.login()

    sys.argv = [
        'train.py', '--wandb', '--wandb_project', 'Connect4RL-othello',
        '--train.total_timesteps', '10000000',
        '--train.eval_interval', '500000',
        '--train.device', 'cuda',
        '--curriculum.phase_4_fraction', '0.60',
        '--curriculum.phase_5_fraction', '0.60',
    ]

    from othello.train import train

    def run_trial():
        train()

    wandb.agent(sweep_id=sweep_id, function=run_trial, count=10)

mp.set_start_method('spawn', force=True)
processes = [mp.Process(target=worker, args=(API_KEY, SWEEP_ID)) for _ in range(3)]
for p in processes:
    p.start()
for p in processes:
    p.join()

print("All trials complete.")